# 3.4 Dynamic Prompts — Per-Request System Prompts

## What you will learn in this notebook

- Why **static system prompts** don't work for multi-user, multi-locale apps
- How **`@dynamic_prompt`** generates the system prompt fresh on every LLM call
- How to use **runtime context** to personalise prompts per user
- How one agent can respond in different languages based on user preference

---

### The problem with static system prompts

A static system prompt is set once at `create_agent()` time and never changes:
```python
agent = create_agent(system_prompt="You are a helpful assistant.")  # Fixed forever
```

But in production, the prompt often needs to vary **per user**:
- User A speaks French → prompt should say "respond in French"
- User B is an admin → prompt should include admin instructions
- User C is a free-tier user → prompt should include limitations

**`@dynamic_prompt`** solves this by running a Python function to generate the system prompt on every LLM call, with access to the current request context.

### `@dynamic_prompt` vs `@wrap_model_call`

| | `@dynamic_prompt` | `@wrap_model_call` |
|---|---|---|
| **Purpose** | Change the system prompt | Change anything (model, tools, prompt, params) |
| **Returns** | A string (the new system prompt) | A ModelResponse (must call handler) |
| **Simpler?** | Yes — purpose-built for prompts | No — more general and verbose |

In [ ]:
# ============================================================
# CELL 1: Load Environment Variables
# ============================================================

from dotenv import load_dotenv

load_dotenv()

In [ ]:
# ============================================================
# CELL 2: Define a Dynamic Prompt Based on User Language
# ============================================================
# Step 1: Define the context schema
#   LanguageContext holds the user's preferred language.
#   Default is "English" — used if no context is injected.
#   This comes from your auth system, user profile DB, etc.
#   at request time — not from the conversation.
#
# Step 2: Define the dynamic prompt function
#   @dynamic_prompt marks this function as a prompt generator.
#   It is called before EVERY LLM call inside the agent loop.
#
#   Function signature:
#     request: ModelRequest → the full LLM call context
#       request.runtime.context → the LanguageContext object
#     returns: str → the system prompt to use for this call
#
#   Logic here:
#     If user speaks English → plain "You are a helpful assistant."
#     If user speaks anything else → append "only respond in {language}."
#
# Why append to a base_prompt rather than writing separate full prompts?
#   DRY (Don't Repeat Yourself): the base instructions stay in one place.
#   Language instruction is a single line added at the end.
#   Easier to maintain as the base prompt evolves.

from dataclasses import dataclass
from langchain.agents.middleware import dynamic_prompt, ModelRequest

@dataclass
class LanguageContext:
    user_language: str = "English"   # Default — override per user at invoke() time

@dynamic_prompt
def user_language_prompt(request: ModelRequest) -> str:
    """Generate system prompt based on user language preference."""

    # Read language from runtime context (injected at invoke() time)
    user_language = request.runtime.context.user_language
    base_prompt = "You are a helpful assistant."

    if user_language != "English":
        # Append language instruction to the base prompt
        return f"{base_prompt} only respond in {user_language}."
    else:
        return base_prompt   # No language modifier for English speakers

In [ ]:
# ============================================================
# CELL 3: Create the Agent With Dynamic Prompt Middleware
# ============================================================
# Notice: NO system_prompt= in create_agent().
# The @dynamic_prompt middleware provides the system prompt dynamically.
# If you also pass a static system_prompt=, the middleware prompt
# will REPLACE it on every call.
#
# context_schema=LanguageContext declares the context type so
# LangChain validates that the context= passed at invoke() time
# matches this schema.

from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    context_schema=LanguageContext,  # Declare expected context type
    middleware=[user_language_prompt]  # Dynamic prompt generator
    # No system_prompt= — the middleware provides it
)

In [ ]:
# ============================================================
# CELL 4: Invoke for an Irish-Speaking User
# ============================================================
# context=LanguageContext(user_language="Irish")
#   → The @dynamic_prompt function reads "Irish" from request.runtime.context
#   → Returns: "You are a helpful assistant. only respond in Irish."
#   → The model responds in Irish (Gaeilge)
#
# Same agent object, same question — completely different response
# because the system prompt changes per-request.

from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="Hello, how are you?")]},
    context=LanguageContext(user_language="Irish")  # Injected per request
)

print(response["messages"][-1].content)

In [ ]:
# ============================================================
# CELL 5: Same Agent, Spanish User
# ============================================================
# Only the context changes — the agent, model, and question are identical.
# Dynamic prompt generates: "You are a helpful assistant. only respond in Spanish."

response = agent.invoke(
    {"messages": [HumanMessage(content="Hello, how are you?")]},
    context=LanguageContext(user_language="Spanish")
)

print(response["messages"][-1].content)

In [ ]:
# ============================================================
# CELL 6: Same Agent, French User
# ============================================================
# The @dynamic_prompt pattern scales to any number of languages
# or conditions without creating separate agents or hardcoding
# language-specific system prompts in multiple places.
#
# In production: `user_language` would come from:
#   - User's browser locale (Accept-Language header)
#   - User profile setting in your database
#   - Explicit user preference stored in session
# All of these would be injected via EmailContext at request time.

response = agent.invoke(
    {"messages": [HumanMessage(content="Hello, how are you?")]},
    context=LanguageContext(user_language="French")
)

print(response["messages"][-1].content)

---
## Summary

| Concept | Key takeaway |
|---|---|
| **`@dynamic_prompt`** | Returns a string that becomes the system prompt for each LLM call |
| **`request.runtime.context`** | Access the context object (e.g. LanguageContext) inside the prompt function |
| **No static `system_prompt=`** | Dynamic prompt replaces it — don't mix both |
| **Per-request isolation** | Each `.invoke()` can pass different context → different prompt → different behaviour |
| **DRY base prompt** | Keep common instructions in a base string, append per-user variations |

### Common dynamic prompt patterns

```python
# Language
f"You are a helpful assistant. Only respond in {context.language}."

# User role
if context.role == "admin":
    return base + " You have access to admin features."

# Subscription tier
if context.plan == "free":
    return base + " Keep responses under 100 words."

# Time of day
if datetime.now().hour < 12:
    return base + " Wish the user a good morning."
```

### Next steps
- **3.4 Dynamic Tools** — restrict which tools are available per user role
- **3.5 Email Agent** — combine dynamic prompts, dynamic tools, and HITL in one app